# Análisis de Canasta de Mercado: FISIMart S.A.C.

Descubrimiento de reglas de asociación **a nivel producto (SKU)** a partir de las
transacciones del datamart, orientado a promociones cruzadas, bundles y recomendaciones
de venta cruzada en tienda física y canal Online.

#### CELDA 1: CONTROL DE ENTORNO E IMPORTACIÓN

In [ ]:
import subprocess
import sys

try:
    import mlxtend  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mlxtend"])

import numpy as np
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth

pd.options.mode.chained_assignment = None

# Rutas relativas desde notebooks/
RUTA_PROCESSED = "../data/processed"
ARCHIVO_FACT_VENTAS = f"{RUTA_PROCESSED}/Fact_Ventas.csv"
ARCHIVO_DIM_PRODUCTO = f"{RUTA_PROCESSED}/Dim_Producto.csv"
ARCHIVO_REGLAS_EXPORT = f"{RUTA_PROCESSED}/Reglas_Asociacion.csv"

SEMILLA = 20
np.random.seed(SEMILLA)

print("=" * 72)
print("     MARKET BASKET ANALYSIS — FISIMart S.A.C. (Reglas de Asociación)")
print("=" * 72)
print(f"● Datamart procesado : {RUTA_PROCESSED}/")
print(f"● Semilla aleatoria  : {SEMILLA}")
print("=" * 72)

     MARKET BASKET ANALYSIS — FISIMart S.A.C. (Reglas de Asociación)
● Datamart procesado : ../data/processed/
● Semilla aleatoria  : 20


#### CELDA 2: CARGA DE DATOS PROCESADOS

In [16]:
Fact_Ventas = pd.read_csv(ARCHIVO_FACT_VENTAS, sep=";", encoding="utf-8")
Dim_Producto = pd.read_csv(ARCHIVO_DIM_PRODUCTO, sep=";", encoding="utf-8")

df_ventas_enriquecido = Fact_Ventas.merge(
    Dim_Producto[["id_producto", "nombre", "categoria", "subcategoria"]],
    on="id_producto",
    how="inner",
)

n_lineas = len(df_ventas_enriquecido)
n_boletas = df_ventas_enriquecido["id_venta"].nunique()
n_categorias = df_ventas_enriquecido["categoria"].nunique()
n_productos = df_ventas_enriquecido["id_producto"].nunique()

print("Resumen inicial del universo analítico:")
print(f"  • Líneas de venta              : {n_lineas:,}")
print(f"  • Boletas únicas (id_venta)    : {n_boletas:,}")
print(f"  • Categorías distintas         : {n_categorias}")
print(f"  • Productos distintos          : {n_productos}")
print(f"\nCategorías disponibles: {sorted(df_ventas_enriquecido['categoria'].unique())}")

Resumen inicial del universo analítico:
  • Líneas de venta              : 25,413
  • Boletas únicas (id_venta)    : 12,750
  • Categorías distintas         : 5
  • Productos distintos          : 553

Categorías disponibles: ['Abarrotes', 'Bebidas', 'Cuidado Personal', 'Hogar', 'Tecnologia']


#### CELDA 3: CONSTRUCCIÓN DE TRANSACCIONES (CANASTA)

**Nivel de análisis elegido: `nombre` (producto / SKU)**

Cada ítem de la canasta es el nombre del producto vendido. Esto permite:
- Detectar **pares concretos de SKUs** que se compran juntos (bundles, cross-sell en checkout).
- Conectar las reglas directamente con el catálogo de `Dim_Producto`.
- Usar `categoria` solo como contexto interpretativo, no como unidad de análisis.

**Filtrado de canastas unitarias**

Se excluyen boletas con un solo producto distinto porque las reglas de asociación requieren
co-ocurrencia entre al menos dos SKUs distintos.

In [17]:
NIVEL_ANALISIS = "nombre"
FILTRAR_CANASTAS_UNITARIAS = True

transacciones_raw = (
    df_ventas_enriquecido
    .groupby("id_venta")[NIVEL_ANALISIS]
    .apply(lambda items: sorted(set(items)))
    .tolist()
)

n_transacciones_raw = len(transacciones_raw)

if FILTRAR_CANASTAS_UNITARIAS:
    transacciones = [t for t in transacciones_raw if len(t) > 1]
else:
    transacciones = transacciones_raw

n_transacciones = len(transacciones)
tamano_promedio = np.mean([len(t) for t in transacciones]) if n_transacciones else 0
n_excluidas = n_transacciones_raw - n_transacciones
n_productos_en_canastas = len({prod for tx in transacciones for prod in tx})

print(f"Nivel de análisis                         : producto (nombre)")
print(f"Transacciones totales (sin filtrar)     : {n_transacciones_raw:,}")
print(f"Canastas unitarias excluidas             : {n_excluidas:,}")
print(f"Transacciones para minería de reglas     : {n_transacciones:,}")
print(f"Productos distintos en canastas          : {n_productos_en_canastas:,}")
print(f"Tamaño promedio de canasta (ítems dist.): {tamano_promedio:.2f}")
print(f"\nEjemplo de transacciones (primeras 5):")
for tx in transacciones[:5]:
    print(f"  {tx}")

Nivel de análisis                         : producto (nombre)
Transacciones totales (sin filtrar)     : 12,750
Canastas unitarias excluidas             : 3,252
Transacciones para minería de reglas     : 9,498
Productos distintos en canastas          : 549
Tamaño promedio de canasta (ítems dist.): 2.33

Ejemplo de transacciones (primeras 5):
  ['Producto Accesorios 112', 'Producto Jabones 4', 'Producto Jugos 111']
  ['Producto Conservas 57', 'Producto Shampoo 221']
  ['Producto Arroz y Menestras 74', 'Producto Cervezas 452', 'Producto Gadgets 299']
  ['Producto Aguas 48', 'Producto Utensilios 32', 'Producto Utensilios 49']
  ['Producto Jabones 27', 'Producto Utensilios 551']


#### CELDA 4: CODIFICACIÓN ONE-HOT DE TRANSACCIONES

In [18]:
encoder = TransactionEncoder()
matriz_bool = encoder.fit(transacciones).transform(transacciones)
df_transacciones = pd.DataFrame(matriz_bool, columns=encoder.columns_)

n_filas, n_columnas = df_transacciones.shape
print(f"Dimensiones de la matriz one-hot: {n_filas} transacciones × {n_columnas} ítems")
print(f"\nColumnas (ítems): {list(df_transacciones.columns)}")
print(f"\nVista previa (5 primeras filas):")
display(df_transacciones.head())

Dimensiones de la matriz one-hot: 9498 transacciones × 549 ítems

Columnas (ítems): ['Producto Accesorios 110', 'Producto Accesorios 112', 'Producto Accesorios 143', 'Producto Accesorios 209', 'Producto Accesorios 216', 'Producto Accesorios 222', 'Producto Accesorios 250', 'Producto Accesorios 278', 'Producto Accesorios 282', 'Producto Accesorios 288', 'Producto Accesorios 290', 'Producto Accesorios 30', 'Producto Accesorios 303', 'Producto Accesorios 325', 'Producto Accesorios 355', 'Producto Accesorios 364', 'Producto Accesorios 38', 'Producto Accesorios 386', 'Producto Accesorios 486', 'Producto Accesorios 498', 'Producto Accesorios 503', 'Producto Accesorios 517', 'Producto Accesorios 548', 'Producto Accesorios 566', 'Producto Accesorios 589', 'Producto Accesorios 82', 'Producto Accesorios 90', 'Producto Accesorios 92', 'Producto Accesorios 94', 'Producto Aceites 114', 'Producto Aceites 137', 'Producto Aceites 178', 'Producto Aceites 186', 'Producto Aceites 197', 'Producto Aceites 

,Producto Accesorios 110,Producto Accesorios 112,Producto Accesorios 143,Producto Accesorios 209,Producto Accesorios 216,Producto Accesorios 222,Producto Accesorios 250,Producto Accesorios 278,Producto Accesorios 282,Producto Accesorios 288,...,Producto Utensilios 49,Producto Utensilios 50,Producto Utensilios 515,Producto Utensilios 532,Producto Utensilios 551,Producto Utensilios 594,Producto Utensilios 73,Producto Utensilios 75,Producto Utensilios 85,Producto Utensilios 88
0,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False


#### CELDA 5: MINERÍA DE ITEMSETS FRECUENTES

**Soporte mínimo:(0,1 %)**

A nivel producto el espacio combinatorio es muy disperso (~550 SKUs). Con ~9 500 canastas
multi-ítem, un soporte del 0,1 % exige al menos ~10 co-compras del par.



In [19]:
MIN_SUPPORT = 0.001


def formatear_itemset(itemset):
    """Convierte el frozenset interno de mlxtend a texto legible."""
    return ", ".join(sorted(itemset))


itemsets_apriori = apriori(
    df_transacciones,
    min_support=MIN_SUPPORT,
    use_colnames=True,
)

itemsets_fpgrowth = fpgrowth(
    df_transacciones,
    min_support=MIN_SUPPORT,
    use_colnames=True,
)

print(f"Itemsets frecuentes — Apriori   : {len(itemsets_apriori)}")
print(f"Itemsets frecuentes — FP-Growth : {len(itemsets_fpgrowth)}")

itemsets_frecuentes = (
    itemsets_apriori
    .sort_values("support", ascending=False)
    .reset_index(drop=True)
    .assign(itemsets=lambda df: df["itemsets"].apply(formatear_itemset))
)

print(f"\nTop itemsets frecuentes (soporte descendente):")
display(itemsets_frecuentes.head(15)[["itemsets", "support"]])

Itemsets frecuentes — Apriori   : 379
Itemsets frecuentes — FP-Growth : 379

Top itemsets frecuentes (soporte descendente):


,itemsets,support
0,Producto Conservas 115,0.053064
1,Producto Jabones 72,0.048747
2,Producto Organizadores 22,0.046852
3,Producto Dental 97,0.044430
4,Producto Cargadores 87,0.043062
5,Producto Cervezas 95,0.039271
6,Producto Jugos 3,0.037797
7,Producto Arroz y Menestras 2,0.037166
8,Producto Utensilios 73,0.035692
9,Producto Arroz y Menestras 74,0.034744


#### CELDA 6: GENERACIÓN Y FILTRADO DE REGLAS DE ASOCIACIÓN

Criterios adaptados al nivel **producto**:
- **Confianza mínima = 0.03 (3 %):** a nivel SKU los pares exactos se repiten menos que a nivel categoría.
- **Lift > 1.05:** el consecuente aparece al menos un 5 % más de lo esperable por azar.
- **Reglas unívocas (1 antecedente → 1 consecuente):** accionables en tienda y Online.

In [29]:
MIN_CONFIDENCE = 0.03
MIN_LIFT = 1.00

mapa_producto_categoria = (
    df_ventas_enriquecido
    .drop_duplicates(subset=["nombre"])
    .set_index("nombre")["categoria"]
    .to_dict()
)


def generar_reglas_producto(df_matriz, min_support, min_confidence, min_lift):
    """Genera reglas de asociación a nivel producto con filtrado y metadatos."""
    itemsets = apriori(df_matriz, min_support=min_support, use_colnames=True)
    if itemsets.empty:
        return pd.DataFrame()

    reglas = association_rules(
        itemsets,
        metric="confidence",
        min_threshold=min_confidence,
    )
    reglas = reglas[
        (reglas["lift"] > min_lift)
        & (reglas["antecedents"].apply(len) == 1)
        & (reglas["consequents"].apply(len) == 1)
    ].copy()

    reglas["antecedents"] = reglas["antecedents"].apply(formatear_itemset)
    reglas["consequents"] = reglas["consequents"].apply(formatear_itemset)
    reglas["categoria_antecedente"] = reglas["antecedents"].map(mapa_producto_categoria)
    reglas["categoria_consecuente"] = reglas["consequents"].map(mapa_producto_categoria)

    return reglas[[
        "antecedents", "consequents",
        "categoria_antecedente", "categoria_consecuente",
        "support", "confidence", "lift",
    ]].sort_values(["lift", "support"], ascending=[False, False]).reset_index(drop=True)


reglas_priorizadas = generar_reglas_producto(
    df_transacciones,
    min_support=MIN_SUPPORT,
    min_confidence=MIN_CONFIDENCE,
    min_lift=MIN_LIFT,
)

print(
    f"Reglas de producto (soporte ≥ {MIN_SUPPORT}, conf ≥ {MIN_CONFIDENCE}, lift > {MIN_LIFT}): "
    f"{len(reglas_priorizadas)}"
)

print(f"\nTop reglas priorizadas (máx. 15):")
display(reglas_priorizadas.head(15))


Reglas de producto (soporte ≥ 0.001, conf ≥ 0.03, lift > 1.0): 38

Top reglas priorizadas (máx. 15):


,antecedents,consequents,categoria_antecedente,categoria_consecuente,support,confidence,lift
0,Producto Cervezas 29,Producto Dental 97,Bebidas,Cuidado Personal,0.001053,0.085470,1.923685
1,Producto Accesorios 92,Producto Accesorios 82,Tecnologia,Tecnologia,0.001053,0.033113,1.776855
2,Producto Accesorios 82,Producto Accesorios 92,Tecnologia,Tecnologia,0.001053,0.056497,1.776855
3,Producto Utensilios 49,Producto Aguas 117,Hogar,Bebidas,0.001158,0.041667,1.576693
4,Producto Aguas 117,Producto Utensilios 49,Bebidas,Hogar,0.001158,0.043825,1.576693
5,Producto Jabones 4,Producto Jugos 101,Cuidado Personal,Bebidas,0.001158,0.042146,1.343285
6,Producto Jugos 101,Producto Jabones 4,Bebidas,Cuidado Personal,0.001158,0.036913,1.343285
7,Producto Gaseosas 54,Producto Aguas 117,Bebidas,Bebidas,0.001053,0.035211,1.332417
8,Producto Aguas 117,Producto Gaseosas 54,Bebidas,Bebidas,0.001053,0.039841,1.332417
9,Producto Dental 97,Producto Cervezas 95,Cuidado Personal,Bebidas,0.002316,0.052133,1.327497


#### CELDA 7: INTERPRETACIÓN Y TRADUCCIÓN A ACCIONES DE NEGOCIO

---

### Regla 1 — Producto Cervezas 29 → Producto Dental 97
| Métrica | Valor | Lectura |
|---------|-------|---------|
| Categorías | Bebidas → Cuidado Personal | Cross-category con mayor lift (~1,92) |
| Soporte | ~0,1 % | Par en ~10 boletas cada 9 500 |
| Confianza | ~8,5 % | Lift alto compensa baja confianza absoluta |

**Acciones:** Exhibidor de cuidado bucal junto a cervezas premium. Bundle fin de semana "Noche en casa". Recomendación en app al agregar cerveza al carrito.

---

### Regla 2 — Producto Accesorios 82 ↔ Producto Accesorios 92
| Métrica | Valor | Lectura |
|---------|-------|---------|
| Categorías | Tecnología → Tecnología | Complementariedad en el mismo pasillo |
| Lift | ~1,78 | Alta afinidad relativa entre accesorios |

**Acciones:** Ubicar ambos SKUs en la misma góndola. Pack "Accesorios tech" con descuento combinado. Upsell en checkout Online.

---

### Regla 3 — Producto Utensilios 49 ↔ Producto Aguas 117
| Métrica | Valor | Lectura |
|---------|-------|---------|
| Categorías | Hogar ↔ Bebidas | Compra de conveniencia multi-pasillo |
| Lift | ~1,58 | Señal estable de canasta mixta |

**Acciones:** Zona de conveniencia cerca de cajas con utensilios + agua. Promoción cruzada en delivery.

---

### Regla 4 — Producto Jabones 4 ↔ Producto Jugos 101
| Métrica | Valor | Lectura |
|---------|-------|---------|
| Categorías | Cuidado Personal ↔ Bebidas | Patrón de compra familiar |
| Lift | ~1,34 | Afinidad moderada-alta entre SKUs concretos |

**Acciones:** Colocar jugos infantiles cerca de jabones. Campaña CRM para hogares con compras recurrentes.

---

### Regla 5 — Producto Gaseosas 54 ↔ Producto Aguas 117
| Métrica | Valor | Lectura |
|---------|-------|---------|
| Categorías | Bebidas ↔ Bebidas | Complemento dentro del pasillo de bebidas |
| Lift | ~1,33 | Quien compra gaseosa también tiende a agua |

**Acciones:** Exhibidor mixto gaseosa + agua. Descuento 2.º producto de bebida al 50 %.

---

### Regla 6 — Producto Dental 97 ↔ Producto Cervezas 95
| Métrica | Valor | Lectura |
|---------|-------|---------|
| Categorías | Cuidado Personal ↔ Bebidas | Mayor soporte entre pares (~0,23 %) |
| Lift | ~1,33 | Regla más frecuente a nivel SKU |

**Acciones:** Cross-merchandising en tiendas de conveniencia. Push post-compra con productos complementarios.


---

### Acciones transversales
1. **Layout:** Reglas mismo pasillo → proximidad física directa.
2. **Online:** Motor de recomendaciones con `reglas_asociacion_fisimart.csv`.
3. **CRM:** Segmentar por pares con lift > 1,5.
4. **Monitoreo:** Recalcular reglas mensualmente.

#### CELDA 8: EXPORTACIÓN PARA POWER BI

In [30]:
columnas_pbi = [
    "antecedents", "consequents",
    "categoria_antecedente", "categoria_consecuente",
    "support", "confidence", "lift",
]

df_export = reglas_priorizadas[columnas_pbi].copy()
df_export["support"] = df_export["support"].round(6)
df_export["confidence"] = df_export["confidence"].round(6)
df_export["lift"] = df_export["lift"].round(6)

df_export.to_csv(ARCHIVO_REGLAS_EXPORT, sep=";", index=False, encoding="utf-8")

print(f"✓ Reglas exportadas: {len(df_export)} filas")
print(f"✓ Archivo generado   : {ARCHIVO_REGLAS_EXPORT}")
print(f"\nVista previa del archivo para Power BI:")
display(df_export.head(15))

✓ Reglas exportadas: 38 filas
✓ Archivo generado   : ../data/processed/reglas_asociacion_fisimart.csv

Vista previa del archivo para Power BI:


,antecedents,consequents,categoria_antecedente,categoria_consecuente,support,confidence,lift
0,Producto Cervezas 29,Producto Dental 97,Bebidas,Cuidado Personal,0.001053,0.085470,1.923685
1,Producto Accesorios 92,Producto Accesorios 82,Tecnologia,Tecnologia,0.001053,0.033113,1.776855
2,Producto Accesorios 82,Producto Accesorios 92,Tecnologia,Tecnologia,0.001053,0.056497,1.776855
3,Producto Utensilios 49,Producto Aguas 117,Hogar,Bebidas,0.001158,0.041667,1.576693
4,Producto Aguas 117,Producto Utensilios 49,Bebidas,Hogar,0.001158,0.043825,1.576693
5,Producto Jabones 4,Producto Jugos 101,Cuidado Personal,Bebidas,0.001158,0.042146,1.343285
6,Producto Jugos 101,Producto Jabones 4,Bebidas,Cuidado Personal,0.001158,0.036913,1.343285
7,Producto Gaseosas 54,Producto Aguas 117,Bebidas,Bebidas,0.001053,0.035211,1.332417
8,Producto Aguas 117,Producto Gaseosas 54,Bebidas,Bebidas,0.001053,0.039841,1.332417
9,Producto Dental 97,Producto Cervezas 95,Cuidado Personal,Bebidas,0.002316,0.052133,1.327497
